In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

In [2]:
def plot_clusters(X_2d, cluster_labels, title, filepath):
    colors = ['darkred', 'darkblue']
    plt.figure()
    for cluster_id in np.unique(cluster_labels):
        plt.scatter(X_2d[cluster_labels == cluster_id, 0], X_2d[cluster_labels == cluster_id, 1],
                    label=f"Cluster {cluster_id}", alpha=0.3, s=10, color=colors[cluster_id % len(colors)])
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(filepath)
    plt.close()

In [3]:
def plot_boundary(X_2d, labels, classifier, title, filepath):
    # Meshgrid for boundary
    h = 0.1
    x_min, x_max = X_2d[:, 0].min() - .5, X_2d[:, 0].max() + .5
    y_min, y_max = X_2d[:, 1].min() - .5, X_2d[:, 1].max() + .5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

    # Predict
    Z = classifier.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    # Plot
    plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.Paired)
    for i, label in enumerate(["Literal", "Metaphorical"]):
        plt.scatter(X_2d[labels == i, 0], X_2d[labels == i, 1], label=label, alpha=0.3, s=10)

    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(filepath)
    plt.close()

In [4]:
def main(models, emb_dir, emb_types, classifiers):
    for model in models:
        print(f"{model}...")
        for emb_type in emb_types:
            print(f"\t{emb_type}...")

            emb_file = f"{emb_dir}/{model}/{emb_type}.npz"
            data = np.load(emb_file)
            embeddings = data["embeddings"]
            labels = data["labels"]

            # Standardize and PCA 2D
            X_scaled = StandardScaler().fit_transform(embeddings)
            X_2d = PCA(n_components=2).fit_transform(X_scaled)

            out_dir = f"{model}/{emb_type}"
            os.makedirs(out_dir, exist_ok=True)

            # Clustering
            kmeans = KMeans(n_clusters=2, random_state=42, n_init=20)
            cluster_labels = kmeans.fit_predict(X_2d)
            plot_title = f"k-means on PCA of {model}/{emb_type}"
            out_file = f"{out_dir}/kmeans_clusters.png"
            plot_clusters(X_2d, cluster_labels, plot_title, out_file)

            # Classification
            for clf_name, classifier in classifiers.items():
                classifier.fit(X_2d, labels)
                plot_title = f"{clf_name} on PCA of {model}/{emb_type}"
                out_file = f"{out_dir}/{clf_name}_boundary.png"
                plot_boundary(X_2d, labels, classifier, plot_title, out_file)

In [5]:
emb_dir = "../2-embeddings"
models = ["bert-base-uncased", "roberta-base"]
emb_types = ["cls", "mean", "layerwise"]
classifiers = {
    "logreg": LogisticRegression(max_iter=1000),
    "svm": SVC(kernel="linear")
}

In [6]:
main(models, emb_dir, emb_types, classifiers)

bert-base-uncased...
	cls...
	mean...
	layerwise...
roberta-base...
	cls...
	mean...
	layerwise...


In [7]:
emb_dir = "../2-embeddings"
models = ["bert-base-uncased_ft", "roberta-base_ft"]
emb_types = ["cls", "mean", "layerwise"]
classifiers = {
    "logreg": LogisticRegression(max_iter=1000),
    "svm": SVC(kernel="linear")
}

In [8]:
main(models, emb_dir, emb_types, classifiers)

bert-base-uncased_ft...
	cls...
	mean...
	layerwise...
roberta-base_ft...
	cls...
	mean...
	layerwise...
